# 01 URDF Combination

In [ ]:
import kinpy as kp

# urdf_path = 'urdfs/ur_description/urdf/ur5_robot.urdf'
# urdf_path = 'urdfs/DH13_right_palm_URDF/urdf/DH13_right_palm_URDF.urdf'
urdf_path = 'urdfs/ur5_dh13_combined.urdf'

txt = open(urdf_path, 'rb').read()
chain = kp.build_chain_from_urdf(txt)
print(chain)
# serial_chain = kp.build_serial_chain_from_urdf(txt, 'ee_link')
# print(serial_chain)

chain.get_joint_parameter_names()

# dir(chain)
# chain.forward_kinematics([0.1])



In [ ]:
import cloudpickle as cpickle

def load_cpkl(path):
    with open(path, 'rb') as f:
        return cpickle.load(f)
    
pkl_path = 'datasets/robot/robot/robot_rawdata.pkl'
data = load_cpkl(pkl_path)
data


In [49]:
import cloudpickle as cpickle

def load_cpkl(path):
    with open(path, 'rb') as f:
        return cpickle.load(f)
    
# pkl_path = 'datasets/robot/robot/predefined_motion_graphs/000320.pkl'
pkl_path = '/home/zhangwenkang/Projects/motion-blender-gs/datasets/robot/ur5_ep4/predefined_motion_graphs/frame_000000.pkl'
data = load_cpkl(pkl_path)
print(data['joints'].shape)
print(data['link_poses'].shape)
print(len(data['joints']))
data

torch.Size([25, 3])
torch.Size([24, 4, 4])
25


{'joints': tensor([[ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0892],
         [-0.1356, -0.0070,  0.0892],
         [-0.0322,  0.3110,  0.3774],
         [-0.0501,  0.6582,  0.1963],
         [-0.1431,  0.6538,  0.1963],
         [-0.1425,  0.6436,  0.2903],
         [-0.1454,  0.7251,  0.2993],
         [-0.1454,  0.7251,  0.2993],
         [-0.1647,  0.8560,  0.3418],
         [-0.1647,  0.8560,  0.3418],
         [-0.1724,  0.8950,  0.3433],
         [-0.1827,  0.9233,  0.3420],
         [-0.1516,  0.8594,  0.3135],
         [-0.1516,  0.8594,  0.3135],
         [-0.1593,  0.8989,  0.3147],
         [-0.1697,  0.9268,  0.3135],
         [-0.1385,  0.8633,  0.2849],
         [-0.1385,  0.8633,  0.2849],
         [-0.1462,  0.9023,  0.2864],
         [-0.1566,  0.9307,  0.2852],
         [-0.1635,  0.7598,  0.3359],
         [-0.1635,  0.7598,  0.3359],
         [-0.1940,  0.7778,  0.3669],
         [-0.2212,  0.7710,  0.4263]], dtype=torch.float16),
 'joint_names': [

# 02 SAPIEN

In [ ]:
import sapien as sapien
from sapien.utils import Viewer
import numpy as np


def main():
    scene = sapien.Scene()  # Create an instance of simulation world (aka scene)
    scene.set_timestep(1 / 100.0)  # Set the simulation frequency

    # NOTE: How to build (rigid bodies) is elaborated in create_actors.py
    scene.add_ground(altitude=0)  # Add a ground
    actor_builder = scene.create_actor_builder()
    actor_builder.add_box_collision(half_size=[0.5, 0.5, 0.5])
    actor_builder.add_box_visual(half_size=[0.5, 0.5, 0.5], material=[1.0, 0.0, 0.0])
    box = actor_builder.build(name="box")  # Add a box
    box.set_pose(sapien.Pose(p=[0, 0, 0.5]))

    # Add some lights so that you can observe the scene
    scene.set_ambient_light([0.5, 0.5, 0.5])
    scene.add_directional_light([0, 1, -1], [0.5, 0.5, 0.5])

    viewer = scene.create_viewer()  # Create a viewer (window)

    # The coordinate frame in Sapien is: x(forward), y(left), z(upward)
    # The principle axis of the camera is the x-axis
    viewer.set_camera_xyz(x=-4, y=0, z=2)
    # The rotation of the free camera is represented as [roll(x), pitch(-y), yaw(-z)]
    # The camera now looks at the origin
    viewer.set_camera_rpy(r=0, p=-np.arctan2(2, 4), y=0)
    viewer.window.set_camera_parameters(near=0.05, far=100, fovy=1)

    while not viewer.closed:  # Press key q to quit
        scene.step()  # Simulate the world
        scene.update_render()  # Update the world to the renderer
        viewer.render()


if __name__ == "__main__":
    main()

In [ ]:
import sapien


def demo(fix_root_link, balance_passive_force):
    scene = sapien.Scene()
    scene.add_ground(0)

    scene.set_ambient_light([0.5, 0.5, 0.5])
    scene.add_directional_light([0, 1, -1], [0.5, 0.5, 0.5])

    viewer = scene.create_viewer()
    viewer.set_camera_xyz(x=-2, y=0, z=1)
    viewer.set_camera_rpy(r=0, p=-0.3, y=0)

    # Load URDF
    loader = scene.create_urdf_loader()
    loader.fix_root_link = fix_root_link
    # robot: sapien.Articulation = loader.load("./my_data/DH13_right_palm_URDF/urdf/DH13_right_palm_URDF.urdf")
    # robot: sapien.Articulation = loader.load("./my_data/ur_description/urdf/ur5_robot.urdf")
    robot: sapien.Articulation = loader.load("./my_data/ur5_dh13_combined.urdf")
    robot.set_root_pose(sapien.Pose([0, 0, 0], [1, 0, 0, 0]))

    # # Set initial joint positions
    # arm_init_qpos = [4.71, 2.84, 0, 0.75, 4.62, 4.48, 4.88]
    # gripper_init_qpos = [0, 0, 0, 0, 0, 0]
    # init_qpos = arm_init_qpos + gripper_init_qpos
    # robot.set_qpos(init_qpos)

    while not viewer.closed:
        for _ in range(4):  # render every 4 steps
            if balance_passive_force:
                qf = robot.compute_passive_force(
                    gravity=True,
                    coriolis_and_centrifugal=True,
                )
                robot.set_qf(qf)
            scene.step()
        scene.update_render()
        viewer.render()


demo(fix_root_link=True, balance_passive_force=True)

In [ ]:
"""
  <!-- Connection Bracket Link -->
  <link name="connection_bracket_link">
    <visual>
      <geometry>
        <!-- 进一步缩小连接件，并调整其局部位置 -->
        <mesh filename="package://connector/UR5_DH13_tutai.STL" scale="0.0004 0.0004 0.0004"/>
      </geometry>
      <origin xyz="0.0 0.0 0.0" rpy="0.0 0.0 0.0"/>
    </visual>
    <collision>
      <geometry>
        <mesh filename="package://connector/UR5_DH13_tutai.STL" scale="0.0004 0.0004 0.0004"/>
      </geometry>
      <origin xyz="0.0 0.0 0.0" rpy="0.0 0.0 0.0"/>
    </collision>
    <inertial>
      <mass value="0.05"/>
      <origin xyz="0.0 0.0 0.0"/>
      <inertia ixx="0.0005" ixy="0.0" ixz="0.0" iyy="0.0005" iyz="0.0" izz="0.0005"/>
    </inertial>
  </link>

  <!-- Connect UR5 end-effector to connection bracket -->
  <joint name="ur5_to_bracket" type="fixed">
    <parent link="ee_link"/>
    <child link="connection_bracket_link"/>
    <origin xyz="0.03 -0.02 0.0" rpy="0.0 0.0 1.5708"/>
  </joint>

    <!-- Connect connection bracket to DH13 hand -->
  <joint name="bracket_to_dh13" type="fixed">
    <parent link="connection_bracket_link"/>
    <child link="right_palm_link"/>
    <!-- 调整DH13手部相对于连接件的位置和方向 -->
    <origin xyz="0.0 0.0 0.015" rpy="0.0 0.0 -1.5708"/>
  </joint>

"""

# 03 Data Process

In [2]:
import os
import cv2
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import sys

base_path = 'my_data/ep3'
output_path = 'datasets/robot/ur5_ep3'
os.makedirs(output_path, exist_ok=True)

video_path = os.path.join(base_path, 'video.mp4')
csv_path = os.path.join(base_path, 'ur5_frame_data.csv')

df = pd.read_csv(csv_path)
df.head()

def extract_and_align(video_path, df, output_dir, save_per_frame_json=False, fallback_fill_last=True):
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"无法打开视频: {video_path}"

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"视频: {video_path}\n尺寸: {width}x{height}, FPS: {fps:.3f}, 帧数: {frame_count}")

    # 准备聚合 JSON 容器
    aligned = {
        'video_path': video_path,
        'csv_path': csv_path,
        'fps': float(fps),
        'frame_count': int(frame_count),
        'frames': []
    }

    idx_map = lambda i: i

    for i in tqdm(range(frame_count), desc='Extracting frames'):
        ret, frame = cap.read()
        assert ret, f"第 {i} 帧读取失败"

        # 保存帧图像
        img_name = f"frame_{i:06d}.png"
        img_dir = os.path.join(output_dir, 'rgb')
        os.makedirs(img_dir, exist_ok=True)
        img_path = os.path.join(img_dir, img_name)
        cv2.imwrite(img_path, frame)

        # 选择 CSV 行
        j = idx_map(i)
        if j >= len(df):
            if fallback_fill_last and len(df) > 0:
                j = len(df) - 1
            else:
                # 无可对齐数据，记录空
                pose_dict = {}
                frame_item = {
                    'frame_index': i,
                    'image': img_path,
                    'pose': pose_dict
                }
                aligned['frames'].append(frame_item)
                if save_per_frame_json:
                    with open(os.path.join(output_dir, f"frame_{i:06d}.json"), 'w', encoding='utf-8') as f:
                        json.dump(frame_item, f, ensure_ascii=False, indent=2)
                continue

        pose_series = df.iloc[j]
        # 确保可序列化
        pose_dict = {k: (None if pd.isna(v) else (float(v) if isinstance(v, (int, float, np.floating)) else v))
                     for k, v in pose_series.to_dict().items()}

        frame_item = {
            'frame_index': i,
            'csv_index': int(j),
            'image': img_path,
            'pose': pose_dict
        }
        aligned['frames'].append(frame_item)

        if save_per_frame_json:
            with open(os.path.join(output_dir, f"frame_{i:06d}.json"), 'w', encoding='utf-8') as f:
                json.dump(frame_item, f, ensure_ascii=False, indent=2)

    cap.release()

    # 写聚合 JSON
    out_json = os.path.join(output_dir, 'frame_pose_alignment.json')
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(aligned, f, ensure_ascii=False, indent=2)

    print(f"Extracted images to: {img_dir}")
    print(f"Extracted aligned JSON: {out_json}")
    return aligned

def prepare_sam2_json(output_path, image_dir, start = 0, end = 499, exp_name='ur5'):
    sam2_json_data = {
        "name": f"{exp_name}:center",
        "instances": [
            f"{exp_name}:kinematic"
        ],
        "root": None,
        "frames": [],
        "output_path": "instance"
    }

    for idx in range(start, end + 1):
        rgb_path = os.path.join(image_dir, f"frame_{idx:06d}.png")
        if os.path.exists(rgb_path):
            sam2_json_data['frames'].append(f'rgb/frame_{idx:06d}.png')
        else:
            print(f"Warning: {rgb_path} does not exist.")

    with open(os.path.join(output_path, 'sam2_task_center.json'), 'w') as f:
        json.dump(sam2_json_data, f, indent=4)
        print(f"Saved SAM2 JSON to {os.path.join(output_path, 'sam2_task_center.json')}")

    return sam2_json_data

def prepare_camera_json(output_path, start = 0, end = 499):
    camera_json = {
        "focal_length": 606.45947266,
        "image_size": [
            640,
            480
        ],
        "orientation": [
            [0.02082761, -0.99729553, 0.07048289],
            [0.10104566, -0.06803744, -0.99255261],
            [0.99466376, 0.02779449, 0.09935532]
        ],
        "position": [
            -0.36250033,
            0.15640983,
            0.7456299
        ],
        "pixel_aspect_ratio": 1.0,
        "principal_point": [
            324.13546753,
            249.23587036
        ],
        "radial_distortion": [
            0.0,
            0.0,
            0.0
        ],
        "skew": 0.0,
        "tangential_distortion": [
            0.0,
            0.0
        ]
    }

    camera_dir = os.path.join(output_path, 'camera')
    os.makedirs(camera_dir, exist_ok=True)

    for idx in range(start, end + 1):
        with open(os.path.join(camera_dir, f"frame_{idx:06d}.json"), 'w') as f:
            json.dump(camera_json, f, indent=4)
    print(f"Saved camera JSON files to {camera_dir}")

    return camera_json

def prepare_depth(video_path, output_path):
    original_cwd = os.getcwd()
    video_depth_anything_path = os.path.join(original_cwd, 'Video-Depth-Anything')
    os.chdir(video_depth_anything_path)
    os.system(f'python3 run.py --input_video ../{video_path} --output_dir ../{output_path} --encoder vits --save_npz')

    os.chdir(original_cwd)

def extract_depth_images(video_path, output_dir):
    npz_path = os.path.join(output_dir, 'depth', 'video_depths.npz')
    if not os.path.exists(npz_path):
        print(f"Depth data not found at {npz_path}. Please run prepare_depth first.")
        prepare_depth(video_path, os.path.join(output_dir, 'depth'))
        
    depth_data = np.load(npz_path, allow_pickle=True)
    depth_images = depth_data['depths']
    depth_dir = os.path.join(output_dir, 'metric_depth')
    os.makedirs(depth_dir, exist_ok=True)
    # save each depth image as npy
    for i, depth_image in enumerate(tqdm(depth_images, desc="Saving depth images as npy")):
        depth_image_path = os.path.join(depth_dir, f"frame_{i:06d}.npy")
        np.save(depth_image_path, depth_image)

In [62]:
# 执行逐帧提取与位姿对齐
aligned = extract_and_align(
    video_path,
    df,
    output_path,
    save_per_frame_json=False,
    fallback_fill_last=True
)

sam2_json = prepare_sam2_json(output_path, 
    os.path.join(output_path, 'rgb'), 
    start=0, 
    end=499, 
    exp_name='ur5'
)

camera_json = prepare_camera_json(output_path, start=0, end=499)


视频: my_data/ep3/video.mp4
尺寸: 640x480, FPS: 30.000, 帧数: 2043


Extracting frames: 100%|██████████| 2043/2043 [00:10<00:00, 202.85it/s]


Extracted images to: datasets/robot/ur5_ep3/rgb
Extracted aligned JSON: datasets/robot/ur5_ep3/frame_pose_alignment.json
Saved SAM2 JSON to datasets/robot/ur5_ep3/sam2_task_center.json
Saved camera JSON files to datasets/robot/ur5_ep3/camera


In [63]:
# Grounded-SAM-2 for instance segmentation
import os
os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + ':Grounded-SAM-2/'
os.system(f'port=8891 ckpt_dir=Grounded-SAM-2/checkpoints bash ./motionblender/preproc/sam2gui/run_gui_app.sh ./{output_path}/sam2_task_center.json')


/home/zhangwenkang/Projects/motion-blender-gs/Grounded-SAM-2/sam2/modeling/sam/transformer.py:23: UserWarning: You are using PyTorch 2.1.2+cu121 without Flash Attention v2 support. Consider upgrading to PyTorch 2.2+ for Flash Attention v2 (which could be faster).
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/home/zhangwenkang/miniconda3/envs/rsrd/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
2025-08-15 14:19:51.444 | INFO     | __main__:init_sam_model:127 - loaded model checkpoint Grounded-SAM-2/checkpoints/sam2.1_hiera_large.pt
2025-08-15 14:19:51.482 | DEBUG    | __main__:set_input_image:191 - Setting frame 0 / 0


* Running on local URL:  http://127.0.0.1:8891

To create a public link, set `share=True` in `launch()`.


2025-08-15 14:19:57.870 | DEBUG    | __main__:select_media:396 - Selected task: ur5:center
/home/zhangwenkang/miniconda3/envs/rsrd/lib/python3.10/site-packages/imageio_ffmpeg/_utils.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
2025-08-15 14:20:01.448 | DEBUG    | __main__:set_input_image:191 - Setting frame 0 / 500
frame loading (JPEG / PNG): 100%|██████████| 500/500 [00:06<00:00, 76.97it/s]
/home/zhangwenkang/Projects/motion-blender-gs/Grounded-SAM-2/sam2/sam2_video_predictor.py:961: UserWarning: cannot import name '_C' from 'sam2' (/home/zhangwenkang/Projects/motion-blender-gs/Grounded-SAM-2/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although s

Keyboard interruption in main thread... closing server.


2

In [64]:
# extract depth images
extract_depth_images(video_path,output_path)

Depth data not found at datasets/robot/ur5_ep3/depth/video_depths.npz. Please run prepare_depth first.


100%|██████████| 93/93 [00:37<00:00,  2.45it/s]
/home/zhangwenkang/miniconda3/envs/rsrd/lib/python3.10/site-packages/imageio_ffmpeg/_utils.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
Saving depth images as npy: 100%|██████████| 2043/2043 [00:01<00:00, 1934.08it/s]


In [65]:
# extract joint poses and link poses

import numpy as np
import kinpy as kp
import torch
import json
import os
import pickle


robot_joints = [
                'world', # fixed
                'shoulder_pan_joint',
                'shoulder_lift_joint',
                'elbow_joint',
                'wrist_1_joint',
                'wrist_2_joint',
                'wrist_3_joint',
                'ee_fixed_joint', # fixed
                'bracket_to_dh13', # fixed
                'right_index_joint_0',
                'right_index_joint_1',
                'right_index_joint_2',
                'right_index_joint_3',
                'right_middle_joint_0',
                'right_middle_joint_1',
                'right_middle_joint_2',
                'right_middle_joint_3',
                'right_ring_joint_0',
                'right_ring_joint_1',
                'right_ring_joint_2',
                'right_ring_joint_3',
                'right_thumb_joint_0',
                'right_thumb_joint_1',
                'right_thumb_joint_2',
                'right_thumb_joint_3'
                ]



robot_joints_from_origin = {                
            'world': 'base_link', # fixed
            'shoulder_pan_joint': 'shoulder_link',
            'shoulder_lift_joint': 'upper_arm_link',
            'elbow_joint': 'forearm_link',
            'wrist_1_joint': 'wrist_1_link',
            'wrist_2_joint': 'wrist_2_link',
            'wrist_3_joint': 'wrist_3_link',
            'ee_fixed_joint': 'ee_link', # fixed
            'bracket_to_dh13': 'right_palm_link', # fixed
            'right_index_joint_0': 'right_index_link_0',
            'right_index_joint_1': 'right_index_link_1',
            'right_index_joint_2': 'right_index_link_2',
            'right_index_joint_3': 'right_index_link_3',
            'right_middle_joint_0': 'right_middle_link_0',
            'right_middle_joint_1': 'right_middle_link_1',
            'right_middle_joint_2': 'right_middle_link_2',
            'right_middle_joint_3': 'right_middle_link_3',
            'right_ring_joint_0': 'right_ring_link_0',
            'right_ring_joint_1': 'right_ring_link_1',
            'right_ring_joint_2': 'right_ring_link_2',
            'right_ring_joint_3': 'right_ring_link_3',
            'right_thumb_joint_0': 'right_thumb_link_0',
            'right_thumb_joint_1': 'right_thumb_link_1',
            'right_thumb_joint_2': 'right_thumb_link_2',
            'right_thumb_joint_3': 'right_thumb_link_3'
        }

robot_connections = [
    ('world', 'shoulder_pan_joint'),
    ('shoulder_pan_joint', 'shoulder_lift_joint'),
    ('shoulder_lift_joint', 'elbow_joint'),
    ('elbow_joint', 'wrist_1_joint'),
    ('wrist_1_joint', 'wrist_2_joint'),
    ('wrist_2_joint', 'wrist_3_joint'),
    ('wrist_3_joint', 'ee_fixed_joint'),
    ('ee_fixed_joint', 'bracket_to_dh13'),

    ('bracket_to_dh13', 'right_index_joint_0'),
    ('right_index_joint_0', 'right_index_joint_1'),
    ('right_index_joint_1', 'right_index_joint_2'),
    ('right_index_joint_2', 'right_index_joint_3'),

    ('bracket_to_dh13', 'right_middle_joint_0'),
    ('right_middle_joint_0', 'right_middle_joint_1'),
    ('right_middle_joint_1', 'right_middle_joint_2'),
    ('right_middle_joint_2', 'right_middle_joint_3'),

    ('bracket_to_dh13', 'right_ring_joint_0'),
    ('right_ring_joint_0', 'right_ring_joint_1'),
    ('right_ring_joint_1', 'right_ring_joint_2'),
    ('right_ring_joint_2', 'right_ring_joint_3'),

    ('bracket_to_dh13', 'right_thumb_joint_0'),
    ('right_thumb_joint_0', 'right_thumb_joint_1'),
    ('right_thumb_joint_1', 'right_thumb_joint_2'),
    ('right_thumb_joint_2', 'right_thumb_joint_3')
]

default_joint_values = {
    'world': 0.0,  # fixed

    'shoulder_pan_joint': -4.4,
    'shoulder_lift_joint': 0.0,
    'elbow_joint': 0.0,
    'wrist_1_joint': 0.0,
    'wrist_2_joint': 0.0,
    'wrist_3_joint': 0.0,

    'ee_fixed_joint': 0.0,  # fixed
    'bracket_to_dh13': 0.0,  # fixed

    'right_index_joint_0': 0.0,
    'right_index_joint_1': 0.17453292519943295,
    'right_index_joint_2': 0.17453292519943295,
    'right_index_joint_3': 0.0,
    'right_middle_joint_0': 0.0,
    'right_middle_joint_1': 0.17453292519943295,
    'right_middle_joint_2': 0.17453292519943295,
    'right_middle_joint_3': 0.0,
    'right_ring_joint_0': 0.0,
    'right_ring_joint_1': 0.17453292519943295,
    'right_ring_joint_2': 0.17453292519943295,
    'right_ring_joint_3': 0.0,
    'right_thumb_joint_0': 0.0,
    'right_thumb_joint_1': 0.0,
    'right_thumb_joint_2': 0.0,
    'right_thumb_joint_3': 0.0
}

joint_map = {
    'q0': 'shoulder_pan_joint',
    'q1': 'shoulder_lift_joint',
    'q2': 'elbow_joint',
    'q3': 'wrist_1_joint',
    'q4': 'wrist_2_joint',
    'q5': 'wrist_3_joint',
}
pose_json_data = {
    'joints': None,
    'joint_names': robot_joints,
    'type': 'kinematic',
    'links': [],
    'link_poses': None
}

robot_connections_int = [(robot_joints.index(a), robot_joints.index(b)) for a, b in robot_connections]
pose_json_data['links'] = robot_connections_int
pose_json_data['joint_names'] = robot_joints


ur5_path="urdfs/ur5_dh13_combined.urdf"
framewise_json_data_path = os.path.join(output_path, 'frame_pose_alignment.json')

output_pkl_path = os.path.join(output_path, 'predefined_motion_graphs')
os.makedirs(output_pkl_path, exist_ok=True)

txt = open(ur5_path, 'rb').read()
ur5_chain = kp.build_chain_from_urdf(txt)
# ur5_chain.get_joint_parameter_names()

with open(framewise_json_data_path, 'r') as f:
    framewise_json_data = json.load(f)
    frame_count = framewise_json_data['frame_count']
    framewise_data_list = framewise_json_data['frames']

for i, frame in enumerate(framewise_data_list):
    joints = torch.zeros((len(robot_joints), 3), dtype=torch.float32)
    link_poses = torch.zeros((len(robot_joints) - 1, 4, 4), dtype=torch.float32)
    for j, joint_name in enumerate(robot_joints):
        cur_pose = frame['pose']
        for k, v in joint_map.items():
            default_joint_values[v] = cur_pose[k]
        cur_link_poses = ur5_chain.forward_kinematics(default_joint_values)
        if j == 0:
            joints[j] = torch.tensor(cur_link_poses['world'].pos)
        else:
            cur_link = robot_joints_from_origin[joint_name]
            joints[j] = torch.tensor(cur_link_poses[cur_link].pos)
            link_poses[j - 1] = torch.tensor(cur_link_poses[cur_link].matrix())

    # print(f"Frame {i}, Joints shape: {joints.shape}, Link poses shape: {link_poses.shape}")
    
    save_path = os.path.join(output_pkl_path, f'frame_{i:06d}.pkl')
    pose_json_data['joints'] = joints
    pose_json_data['link_poses'] = link_poses
    with open(save_path, 'wb') as f:
        pickle.dump(pose_json_data, f)
    # break



In [4]:
# prepare dataset.json
import json
import os

rgb_dir = os.path.join(output_path, 'rgb')
rgb_list = os.listdir(rgb_dir)[:200]
json_data = {}
json_data['count'] = len(rgb_list)
json_data['ids'] = [f'frame_{i:06d}' for i in range(len(rgb_list))]

with open(os.path.join(output_path, 'dataset.json'), 'w') as json_file:
    json.dump(json_data, json_file, indent=4)


# 04 Others

In [ ]:
# Convert a sequence of images into a video using OpenCV

import cv2
import os

def images_to_video(image_folder, video_name, fps=30):
    images = [img for img in os.listdir(image_folder) if img.endswith(".jpg") or img.endswith(".png")]
    images.sort()  # Sort images by name

    if not images:
        print("No images found in the specified folder.")
        return

    first_image = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = first_image.shape

    video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    for image in images:
        video.write(cv2.imread(os.path.join(image_folder, image)))

    video.release()
    print(f"Video {video_name} created successfully.")

if __name__ == "__main__":
    image_folder = 'datasets/robot/robot/rgb'  # Path to the images folder
    video_name = 'output_video.mp4'  # Output video file name
    images_to_video(image_folder, video_name)

In [ ]:
# load npy file
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter


def load_npy(path):
    return np.load(path, allow_pickle=True)

# npy_path = 'datasets/robot/robot/metric_depth/026560.npy'
npy_path = 'datasets/robot/ur5_ep4/aligned_depth_anything/frame_002030.npy'
data = load_npy(npy_path)
print(data.shape)

masked_data = np.ma.masked_where(data > 10000, data)

# 对原始深度数据进行中值滤波
filtered_data = median_filter(data, size=3)
filtered_masked_data = np.ma.masked_where(filtered_data > 10000, filtered_data)

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[0].imshow(masked_data, cmap='plasma')
axs[0].set_title('Original Depth Map')
axs[0].axis('off')
axs[1].imshow(filtered_masked_data, cmap='plasma')
axs[1].set_title('Median Filtered Depth Map')
axs[1].axis('off')
plt.colorbar(axs[1].images[0], ax=axs, label='Depth', orientation='vertical', fraction=0.02)
plt.show()